In [1]:
# ─────────────────────────── standard libs ────────────────────────────────
import os
from pathlib import Path
from collections import Counter
import logging
import warnings
import math

# ────────────────────────────── data stack ────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

# ────────────────────────────── sklearn ───────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix,
)
from sklearn.model_selection import train_test_split

# ─────────────────────────────── torch ────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, TensorDataset, DataLoader

# ─────────────────────────────── other ────────────────────────────────────
import mne
from tqdm import tqdm

mne.set_log_level("ERROR")
logging.getLogger("mne").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [2]:
edf_path = '/kaggle/input/datasets/iamlokigodofmischief/tuh-eeg/tuh_eeg_epilepsy/tuh_eeg_epilepsy/00_epilepsy/aaaaaanr/s001_2003/02_tcp_le/aaaaaanr_s001_t001.edf'
raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
print(raw.info['sfreq'], 'Hz,', len(raw.ch_names), 'channels,',
      raw.n_times / raw.info['sfreq'] / 60, 'minutes long')

250.0 Hz, 33 channels, 20.233333333333334 minutes long


In [3]:
"""
One-pass census for the TUH EEG Epilepsy subset
==============================================

• Walks every .edf only **once**  
• Collects three different histograms:
      1. “How many channels per file?”          → count_hist
      2. “Which sampling rates occur?”          → rate_hist
      3. “How often does each channel label appear?” → label_hist
"""

ROOT = Path("/kaggle/input/datasets/iamlokigodofmischief/tuh-eeg/tuh_eeg_epilepsy/tuh_eeg_epilepsy")   # change if needed

# ------------------------------------------------------------------#
#  Helper: normalise raw Nicolet labels → plain 10-20 names
# ------------------------------------------------------------------#
def clean_label(ch):
    """
    Examples:
      'EEG FP1-LE'  -> 'FP1'
      'EEG CZ-REF'  -> 'CZ'
      'EKG1'        -> 'EKG1' (unchanged)
    """
    if ch.startswith("EEG "):
        ch = ch.split()[1]          # drop 'EEG '
    return ch.split("-")[0]         # drop reference suffix

# ------------------------------------------------------------------#
#  Pass 1: scan every EDF file
# ------------------------------------------------------------------#
count_hist = Counter()    # key = n_channels   (int),  value = #files
rate_hist  = Counter()    # key = sfreq (float),       value = #files
label_hist = Counter()    # key = channel label (str), value = #files
tot_files  = 0

for edf in ROOT.rglob("*.edf"):
    raw = mne.io.read_raw_edf(edf, preload=False, verbose=False)
    tot_files += 1

    # 1) how many channels in *this* file
    count_hist[len(raw.ch_names)] += 1

    # 2) sampling rate (assume one rate per file)
    rate_hist[raw.info["sfreq"]] += 1

    # 3) per-label presence
    for ch in raw.ch_names:
        label_hist[clean_label(ch)] += 1

print(f"\nScanned {tot_files:,} EDF files\n")

# ------------------------------------------------------------------#
#  Pretty tables
# ------------------------------------------------------------------#
# ------------------------------------------------------------------
#  tidy-up helper (safe for both numeric & string indexes)
# ------------------------------------------------------------------
def pretty(counter, idx_name, col_name, numeric=False):
    """
    Convert a Counter → DataFrame and sort it nicely.

    numeric=True  → sort by the index column as integers / floats
    numeric=False → sort descending by the count column
    """
    df = (pd.Series(counter)
            .rename_axis(idx_name)
            .reset_index(name=col_name))

    if numeric:
        df[idx_name] = pd.to_numeric(df[idx_name])
        df = df.sort_values(idx_name)             # ascending numeric
    else:
        df = df.sort_values(col_name, ascending=False)

    return df.reset_index(drop=True)

# ------------------------------------------------------------------
#  pretty tables
# ------------------------------------------------------------------
df_counts = pretty(count_hist, "n_channels", "n_files", numeric=True)
print("Files per channel-count\n" + df_counts.to_string(index=False))

df_rates  = pretty(rate_hist , "Hz",         "n_files", numeric=True)
print("\nSampling-rate distribution\n" + df_rates.to_string(index=False))

df_labels = (pretty(label_hist, "channel", "n_files")      # sort by n_files desc.
               .assign(percent=lambda d: 100*d.n_files/tot_files))
print("\nChannel presence (top 30)\n" +
      df_labels.head(30).to_string(index=False,
                                   formatters={"percent": "{:.2f}%".format}))



Scanned 2,298 EDF files

Files per channel-count
 n_channels  n_files
         17        2
         18        6
         27      158
         28      106
         29      101
         30      229
         31      179
         32      203
         33      260
         34      595
         35       19
         36      314
         41      126

Sampling-rate distribution
    Hz  n_files
 250.0      681
 256.0     1354
 400.0       75
 512.0      168
1000.0       20

Channel presence (top 30)
channel  n_files percent
    FP1     2298 100.00%
    FP2     2298 100.00%
     F3     2298 100.00%
     F4     2298 100.00%
     C3     2298 100.00%
     C4     2298 100.00%
     P3     2298 100.00%
     P4     2298 100.00%
     O1     2298 100.00%
     O2     2298 100.00%
     F7     2298 100.00%
     F8     2298 100.00%
     T3     2298 100.00%
     T4     2298 100.00%
     T5     2298 100.00%
     T6     2298 100.00%
     CZ     2298 100.00%
     PZ     2290  99.65%
     FZ     2290  99.65%
     

In [4]:
ROOT = Path("/kaggle/input/datasets/iamlokigodofmischief/tuh-eeg/tuh_eeg_epilepsy/tuh_eeg_epilepsy")   # change if needed

def build_index(root: Path) -> pd.DataFrame:
    rows = []
    
    # iterate over the two top-level class folders
    for class_dir in ('00_epilepsy', '01_no_epilepsy'):
        label = 'epilepsy' if class_dir.startswith('00') else 'control'
        group_path = root / class_dir
        
        # patient → session → montage → *.edf
        for patient_dir in group_path.iterdir():
            pid = patient_dir.name
            for session_dir in patient_dir.iterdir():
                sid = session_dir.name
                for montage_dir in session_dir.iterdir():
                    mid = montage_dir.name
                    for edf in montage_dir.glob('*.edf'):
                        rows.append({
                            'label'   : label,           # epilepsy / control
                            'patient' : pid,
                            'session' : sid,
                            'montage' : mid,
                            'edf_path': str(edf)
                        })
    
    return pd.DataFrame(rows)
df = build_index(ROOT)

print(f"Indexed {len(df):,} EDF files")
print(df.head())

Indexed 2,298 EDF files
      label   patient    session    montage  \
0  epilepsy  aaaaakvr  s001_2010  01_tcp_ar   
1  epilepsy  aaaaaklv  s003_2011  01_tcp_ar   
2  epilepsy  aaaaaklv  s002_2011  01_tcp_ar   
3  epilepsy  aaaaaklv  s001_2010  02_tcp_le   
4  epilepsy  aaaaaawu  s004_2011  01_tcp_ar   

                                            edf_path  
0  /kaggle/input/datasets/iamlokigodofmischief/tu...  
1  /kaggle/input/datasets/iamlokigodofmischief/tu...  
2  /kaggle/input/datasets/iamlokigodofmischief/tu...  
3  /kaggle/input/datasets/iamlokigodofmischief/tu...  
4  /kaggle/input/datasets/iamlokigodofmischief/tu...  


In [5]:
df['label'].value_counts()          # class balance
df.groupby('patient').size()        # EDF files per subject
df[df['label']=='epilepsy'].head()  # peek at positives

,label,patient,session,montage,edf_path
0,epilepsy,aaaaakvr,s001_2010,01_tcp_ar,/kaggle/input/datasets/iamlokigodofmischief/tu...
1,epilepsy,aaaaaklv,s003_2011,01_tcp_ar,/kaggle/input/datasets/iamlokigodofmischief/tu...
2,epilepsy,aaaaaklv,s002_2011,01_tcp_ar,/kaggle/input/datasets/iamlokigodofmischief/tu...
3,epilepsy,aaaaaklv,s001_2010,02_tcp_le,/kaggle/input/datasets/iamlokigodofmischief/tu...
4,epilepsy,aaaaaawu,s004_2011,01_tcp_ar,/kaggle/input/datasets/iamlokigodofmischief/tu...


In [6]:
# ─── config ──────────────────────────────────────────────────────────────────
NUM_PATIENTS = 60
ROOT         = Path("/kaggle/input/datasets/iamlokigodofmischief/tuh-eeg/"
                    "tuh_eeg_epilepsy/tuh_eeg_epilepsy")

# 21 standard 10-20 channels — must have at least MIN_CHANS present
TARGET_CHANS = ['FP1','FP2','F3','F4','C3','C4','P3','P4','O1','O2',
                'F7','F8','T3','T4','T5','T6','FZ','CZ','PZ','A1','A2']
MIN_CHANS    = 16          # skip file if fewer than this many targets found

FS_TARGET    = 128         # resample everything to 128 Hz
WIN_SEC      = 4           # window length in seconds
WIN_SAMPLES  = FS_TARGET * WIN_SEC          # 512 samples
AMP_THRESH   = 150e-6      # ±150 µV — reject artifact windows  (in Volts)
STEP_SEC     = 2           # 50 % overlap between windows
STEP_SAMPLES = FS_TARGET * STEP_SEC         # 256


# ─── helpers ──────────────────────────────────────────────────────────────────
def patient_total_bytes(patient_dir: Path) -> int:
    return sum(edf.stat().st_size for edf in patient_dir.rglob("*.edf"))

def clean_name(ch: str) -> str:
    """'EEG FP1-LE'  →  'FP1' """
    if ch.startswith("EEG "):
        ch = ch.split()[1]
    return ch.split("-")[0]


def tidy_raw(raw: mne.io.BaseRaw) -> mne.io.BaseRaw | None:
    """
    Full SOTA preprocessing pipeline on a single EDF file.

    Steps
    -----
    1. Rename channels to plain 10-20 labels
    2. Keep only TARGET_CHANS that actually exist  (no zero-padding)
    3. Skip file if fewer than MIN_CHANS are present
    4. Set average reference
    5. Bandpass 1–40 Hz  (FIR, zero-phase)
    6. Notch 60 Hz  (US power-line)
    7. Resample to FS_TARGET
    8. Per-channel z-score normalisation

    Returns None if the file is unusable (too few channels).
    """
    # 1. rename
    raw.rename_channels({c: clean_name(c) for c in raw.ch_names}, verbose=False)

    # 2. keep only those target channels that are actually present
    available = [c for c in TARGET_CHANS if c in raw.ch_names]
    if len(available) < MIN_CHANS:
        return None                       # skip — not enough channels
    raw.pick(available)
    raw.reorder_channels(sorted(available, key=TARGET_CHANS.index))

    # 3. average reference
    raw.set_eeg_reference("average", verbose=False)

    # 4. bandpass 1–40 Hz
    raw.filter(1., 40., fir_design="firwin", verbose=False)

    # 5. notch 60 Hz
    raw.notch_filter(60., fir_design="firwin", verbose=False)

    # 6. resample
    if raw.info["sfreq"] != FS_TARGET:
        raw.resample(FS_TARGET, npad="auto", verbose=False)

    return raw


def load_windows(edf_path: Path):
    """
    Load one EDF → list of (n_channels, WIN_SAMPLES) float32 windows.

    Windows are extracted with 50 % overlap.
    Amplitude-threshold rejection: any window with |value| > AMP_THRESH
    is discarded.
    Per-channel z-score is applied per-window (not globally).
    """
    try:
        raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    except Exception:
        return [], []

    raw = tidy_raw(raw)
    if raw is None:
        return [], []

    data = raw.get_data().astype(np.float32)  # (n_ch, n_times)  in Volts
    n_ch, n_times = data.shape
    stem = edf_path.stem

    windows, names = [], []
    for start in range(0, n_times - WIN_SAMPLES + 1, STEP_SAMPLES):
        win = data[:, start: start + WIN_SAMPLES]          # (n_ch, 512)

        # amplitude rejection
        if np.any(np.abs(win) > AMP_THRESH):
            continue

        # per-channel z-score (avoids global scale issues)
        mu  = win.mean(axis=1, keepdims=True)
        std = win.std(axis=1, keepdims=True) + 1e-8
        win = (win - mu) / std

        windows.append(win)                                # (n_ch, 512)
        names.append(f"{stem}_{start//FS_TARGET:07d}s")

    return windows, names


# ─── step 1  compute patient sizes ───────────────────────────────────────────
ep_patients = sorted((ROOT / "00_epilepsy").iterdir(),    key=patient_total_bytes)
ne_patients = sorted((ROOT / "01_no_epilepsy").iterdir(), key=patient_total_bytes)

ep_sizes = [(p, patient_total_bytes(p)) for p in ep_patients]
ne_sizes = [(p, patient_total_bytes(p)) for p in ne_patients]

# ─── step 2  greedy pairing ───────────────────────────────────────────────────
pairs, used_ne = [], set()
for ep_path, ep_sz in ep_sizes:
    best = None
    for ne_path, ne_sz in ne_sizes:
        if ne_path in used_ne:
            continue
        diff = abs(ep_sz - ne_sz)
        if best is None or diff < best[1]:
            best = (ne_path, diff)
    if best is None:
        break
    used_ne.add(best[0])
    pairs.append((ep_path, best[0]))
    if len(pairs) == NUM_PATIENTS:
        break

ep_sel = [p for p, _ in pairs]
ne_sel = [p for _, p in pairs]

print("🗂  Selected patients (balanced sets)")
for e, n in pairs:
    print(f"   {e.name:<12} ↔ {n.name:<12}  "
          f"({patient_total_bytes(e)/1e6:.1f} MB vs {patient_total_bytes(n)/1e6:.1f} MB)")

# ─── step 3  extract windows ──────────────────────────────────────────────────
all_windows, all_names, all_labels = [], [], []

def process_group(group, lab):
    pbar = tqdm(group, desc=f"{lab}", unit="patient")
    for patient_dir in pbar:
        for edf in patient_dir.rglob("*.edf"):
            wins, names = load_windows(edf)
            all_windows.extend(wins)
            all_names.extend(names)
            all_labels.extend([lab] * len(wins))
        pbar.set_postfix(windows=len(all_windows))

process_group(ep_sel, "epilepsy")
process_group(ne_sel, "non-epilepsy")

# ─── step 4  build arrays ────────────────────────────────────────────────────
# all_windows is a list of (n_ch, WIN_SAMPLES) arrays — channels may vary!
# Pad to TARGET_CHANS length so every sample is (21, 512)
N_CH = len(TARGET_CHANS)   # 21

def pad_to_target(win):
    """Zero-pad channels axis to N_CH if file had fewer channels."""
    n = win.shape[0]
    if n == N_CH:
        return win
    out = np.zeros((N_CH, WIN_SAMPLES), dtype=np.float32)
    out[:n] = win
    return out

X_raw  = np.stack([pad_to_target(w) for w in all_windows]).astype(np.float32)
# shape: (N_windows, 21, 512)
y_raw  = np.array([1 if l == "epilepsy" else 0 for l in all_labels], dtype=np.int64)
names  = np.array(all_names)

print(f"\n✅  Total windows : {len(X_raw):,}")
print(f"   epilepsy      : {(y_raw==1).sum():,}")
print(f"   non-epilepsy  : {(y_raw==0).sum():,}")
print(f"   X shape       : {X_raw.shape}   dtype: {X_raw.dtype}")

🗂  Selected patients (balanced sets)
   aaaaaphz     ↔ aaaaaojm      (17.1 MB vs 17.2 MB)
   aaaaanzp     ↔ aaaaapuo      (17.2 MB vs 17.0 MB)
   aaaaanfl     ↔ aaaaanvy      (17.2 MB vs 16.9 MB)
   aaaaanhr     ↔ aaaaanuz      (17.9 MB vs 17.9 MB)
   aaaaaodl     ↔ aaaaaoxx      (17.9 MB vs 18.5 MB)
   aaaaapwd     ↔ aaaaaoyh      (17.9 MB vs 18.8 MB)
   aaaaapvx     ↔ aaaaanwp      (18.9 MB vs 19.1 MB)
   aaaaanmf     ↔ aaaaakpl      (19.3 MB vs 19.5 MB)
   aaaaalof     ↔ aaaaammt      (19.9 MB vs 19.9 MB)
   aaaaamor     ↔ aaaaaoep      (19.9 MB vs 19.9 MB)
   aaaaapge     ↔ aaaaaoyn      (19.9 MB vs 20.0 MB)
   aaaaamrq     ↔ aaaaamrw      (20.1 MB vs 20.1 MB)
   aaaaakgy     ↔ aaaaalsc      (20.2 MB vs 20.2 MB)
   aaaaamlm     ↔ aaaaammn      (20.6 MB vs 20.4 MB)
   aaaaalhr     ↔ aaaaaofk      (21.1 MB vs 21.1 MB)
   aaaaaoou     ↔ aaaaapgp      (21.9 MB vs 21.9 MB)
   aaaaakyc     ↔ aaaaaocx      (22.1 MB vs 22.1 MB)
   aaaaapjn     ↔ aaaaaooz      (22.6 MB vs 22.9 MB)
   aaaaam

non-epilepsy: 100%|██████████| 60/60 [04:09<00:00,  4.16s/patient, windows=183795]



✅  Total windows : 183,795
   epilepsy      : 84,331
   non-epilepsy  : 99,464
   X shape       : (183795, 21, 512)   dtype: float32


In [7]:
# ─── patient-level split (70 / 15 / 15) ──────────────────────────────────────
# Extract patient ID from window name (first underscore-delimited token)
patient_ids = np.array([n.split("_")[0] for n in names])

patients_ep = sorted(set(patient_ids[y_raw == 1]))
patients_ne = sorted(set(patient_ids[y_raw == 0]))

rng = np.random.default_rng(seed=42)
rng.shuffle(patients_ep);  rng.shuffle(patients_ne)

def split_ids(ids):
    n = len(ids)
    n_tr = int(round(0.70 * n));  n_va = int(round(0.15 * n))
    return list(ids[:n_tr]), list(ids[n_tr:n_tr+n_va]), list(ids[n_tr+n_va:])

ep_train, ep_val, ep_test = split_ids(patients_ep)
ne_train, ne_val, ne_test = split_ids(patients_ne)

train_pids = set(ep_train + ne_train)
val_pids   = set(ep_val   + ne_val)
test_pids  = set(ep_test  + ne_test)

tr_mask   = np.array([p in train_pids for p in patient_ids])
val_mask  = np.array([p in val_pids   for p in patient_ids])
test_mask = np.array([p in test_pids  for p in patient_ids])

X_train, y_train = X_raw[tr_mask],   y_raw[tr_mask]
X_val,   y_val   = X_raw[val_mask],  y_raw[val_mask]
X_test,  y_test  = X_raw[test_mask], y_raw[test_mask]

def summary(X, y, name):
    print(f"  {name:>6}: {len(X):>6,} windows  "
          f"| epi={y.sum():,}  non={(y==0).sum():,}")

print("📊  Split summary (no SMOTE — using weighted sampler instead)")
summary(X_train, y_train, "train")
summary(X_val,   y_val,   "val")
summary(X_test,  y_test,  "test")

# ─── DataLoaders ─────────────────────────────────────────────────────────────
# Instead of SMOTE, use WeightedRandomSampler — correct for time-series data
from torch.utils.data import WeightedRandomSampler

BATCH_SIZE = 32

def make_loader(X, y, weighted=False):
    ds = TensorDataset(
        torch.from_numpy(X),               # float32, shape (N, 21, 512)
        torch.from_numpy(y),               # long
    )
    if weighted:
        counts  = np.bincount(y)
        weights = 1.0 / counts[y]          # inverse-frequency per sample
        sampler = WeightedRandomSampler(
            torch.from_numpy(weights).float(),
            num_samples=len(weights), replacement=True
        )
        return DataLoader(ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True)

train_loader = make_loader(X_train, y_train, weighted=True)
val_loader   = make_loader(X_val,   y_val)
test_loader  = make_loader(X_test,  y_test)

# sanity check
xb, yb = next(iter(train_loader))
print(f"\n✅  Batch — X: {xb.shape}  y: {yb.shape}  dtype: {xb.dtype}")
print(f"   Class balance in batch: epi={yb.sum().item()}  non={BATCH_SIZE - yb.sum().item()}")

📊  Split summary (no SMOTE — using weighted sampler instead)
   train: 124,992 windows  | epi=55,985  non=69,007
     val: 32,944 windows  | epi=14,438  non=18,506
    test: 25,859 windows  | epi=13,908  non=11,951

✅  Batch — X: torch.Size([32, 21, 512])  y: torch.Size([32])  dtype: torch.float32
   Class balance in batch: epi=17  non=15


In [8]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ─────────────────────────────────────────────────────────────────────────────
# EEGformer components
# ─────────────────────────────────────────────────────────────────────────────

def trunc_normal(tensor, mean=0., std=1., a=-2., b=2.):
    """Truncated normal initialiser."""
    def norm_cdf(x):
        return (1. + math.erf(x / math.sqrt(2.))) / 2.

    with torch.no_grad():
        l = norm_cdf((a - mean) / std)
        u = norm_cdf((b - mean) / std)
        tensor.uniform_(2 * l - 1, 2 * u - 1)
        tensor.erfinv_()
        tensor.mul_(std * math.sqrt(2.))
        tensor.add_(mean)
        tensor.clamp_(min=a, max=b)
        return tensor


class Mlp(nn.Module):
    """Feed-forward block used inside every transformer block."""
    def __init__(self, in_features, hidden_features=None, out_features=None,
                 act_layer=nn.GELU, drop=0.):
        super().__init__()
        out_features     = out_features or in_features
        hidden_features  = hidden_features or in_features
        self.fc1  = nn.Linear(in_features, hidden_features)
        self.act  = act_layer()
        self.fc2  = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class GenericTFB(nn.Module):
    """Transformer block used in RTM and STM."""
    def __init__(self, emb_size, num_heads):
        super().__init__()
        self.M  = emb_size
        self.hA = num_heads
        self.Dh = emb_size // num_heads
        self.Wqkv = nn.Parameter(torch.randn(3, num_heads, self.Dh, emb_size))
        self.Wo   = nn.Parameter(torch.randn(emb_size, emb_size))
        self.ln1  = nn.LayerNorm(emb_size)
        self.ln2  = nn.LayerNorm(emb_size)
        self.mlp  = Mlp(emb_size, emb_size * 4)

    def forward(self, x, z):
        # z : (S, C+1, D)  or  (C+1, S+1, D)
        qkv = torch.einsum('xhdm,ijm->xijhd', self.Wqkv, self.ln1(z))
        q, k, v = qkv[0], qkv[1], qkv[2]
        att = (q.transpose(1, 2) / math.sqrt(self.Dh)) @ k.transpose(1, 2).transpose(-2, -1)
        att = torch.softmax(att, dim=-1)
        imv = (att @ v.transpose(1, 2)).transpose(1, 2)
        z = torch.einsum('nm,ijm->ijn', self.Wo,
                         imv.reshape(z.shape[0], z.shape[1], self.M)) + z
        z = self.mlp(self.ln2(z)) + z
        return z


class TemporalTFB(nn.Module):
    """Transformer block used in TTM.
    
    Operates on z of shape (M+1, M) — the temporal token sequence.
    The original x (C+1, S+1, D) is passed in but not used here;
    kept for API consistency with GenericTFB.
    """
    def __init__(self, emb_size, num_heads, avgf):
        super().__init__()
        self.M    = emb_size   # token embedding dim = num_submatrices (M)
        self.hA   = num_heads
        self.Dh   = emb_size // num_heads
        self.avgf = avgf
        self.Wqkv = nn.Parameter(torch.randn(3, num_heads, self.Dh, emb_size))
        self.Wo   = nn.Parameter(torch.randn(emb_size, emb_size))
        self.ln1  = nn.LayerNorm(emb_size)
        self.ln2  = nn.LayerNorm(emb_size)
        self.mlp  = Mlp(emb_size, emb_size * 4)

    def forward(self, x, z):
        # x is unused (kept for API consistency); z : (M+1, M)
        qkv = torch.einsum('xhdm,im->xihd', self.Wqkv, self.ln1(z))
        q, k, v = qkv[0], qkv[1], qkv[2]
        att = (q.transpose(0, 1) / math.sqrt(self.Dh)) @ k.transpose(0, 1).transpose(-2, -1)
        att = torch.softmax(att, dim=-1)
        imv = (att @ v.transpose(0, 1)).transpose(0, 1)
        z = torch.einsum('nm,im->in', self.Wo,
                         imv.reshape(self.avgf + 1, self.M)) + z
        z = self.mlp(self.ln2(z)) + z
        return z


# ─── Depth-wise 1-D Convolutional Module (ODCM) ──────────────────────────────
class ODCM(nn.Module):
    """Three-layer depthwise-conv feature extractor.
    
    Input  : (C, T)
    Output : (C, ncf, S)   where S = T - 3*(kernel_size-1)
    """
    def __init__(self, input_channels, kernel_size=10):
        super().__init__()
        self.ncf = 120
        C = input_channels
        self.cv1  = nn.Conv1d(C, C,          kernel_size, groups=C, padding='valid')
        self.cv2  = nn.Conv1d(C, C,          kernel_size, groups=C, padding='valid')
        self.cv3  = nn.Conv1d(C, self.ncf * C, kernel_size, groups=C, padding='valid')
        self.relu = nn.ReLU()

    def forward(self, x):           # (C, T)
        x = self.relu(self.cv1(x))  # (C, T-k+1)
        x = self.relu(self.cv2(x))  # (C, T-2*(k-1))
        x = self.relu(self.cv3(x))  # (ncf*C, S)
        S = x.shape[1]
        C = x.shape[0] // self.ncf
        return x.reshape(C, self.ncf, S)   # (C, ncf=D, S)


# ─── Regional Transformer Module (RTM) ───────────────────────────────────────
class RTM(nn.Module):
    """
    Input  : (C, D, S)   from ODCM
    Output : (S, C+1, D)
    """
    def __init__(self, odcm_out_shape, num_blocks, num_heads):
        super().__init__()
        C, D, S = odcm_out_shape
        self.M  = D
        self.tK = num_blocks
        self.weight = nn.Parameter(torch.randn(D, D))
        self.bias   = nn.Parameter(torch.zeros(S, C + 1, D))
        self.cls    = nn.Parameter(torch.zeros(S, 1, D))
        trunc_normal(self.bias, std=.02)
        trunc_normal(self.cls,  std=.02)
        self.tfbs = nn.ModuleList([GenericTFB(D, num_heads) for _ in range(num_blocks)])

    def forward(self, x):           # x : (C, D, S)
        # FIX: single permute only — no redundant transposes
        x = x.permute(2, 0, 1)                              # (S, C, D)
        z = torch.einsum('lm,ijm->ijl', self.weight, x)     # (S, C, D)
        z = torch.cat([self.cls, z], dim=1)                  # (S, C+1, D)
        z = z + self.bias
        for tfb in self.tfbs:
            z = tfb(x, z)
        return z                                             # (S, C+1, D)


# ─── Synchronous Transformer Module (STM) ────────────────────────────────────
class STM(nn.Module):
    """
    Input  : (S, C+1, D)   from RTM
    Output : (C+1, S+1, D)
    """
    def __init__(self, rtm_out_shape, num_blocks, num_heads):
        super().__init__()
        S, Cp1, D = rtm_out_shape
        self.M  = D
        self.tK = num_blocks
        self.weight = nn.Parameter(torch.randn(D, D))
        self.bias   = nn.Parameter(torch.zeros(Cp1, S + 1, D))
        self.cls    = nn.Parameter(torch.zeros(Cp1, 1, D))
        trunc_normal(self.bias, std=.02)
        trunc_normal(self.cls,  std=.02)
        self.tfbs = nn.ModuleList([GenericTFB(D, num_heads) for _ in range(num_blocks)])

    def forward(self, x):           # x : (S, C+1, D)
        x = x.transpose(0, 1)                               # (C+1, S, D)
        z = torch.einsum('lm,ijm->ijl', self.weight, x)     # (C+1, S, D)
        z = torch.cat([self.cls, z], dim=1)                  # (C+1, S+1, D)
        z = z + self.bias
        for tfb in self.tfbs:
            z = tfb(x, z)
        return z                                             # (C+1, S+1, D)


# ─── Temporal Transformer Module (TTM) ───────────────────────────────────────
class TTM(nn.Module):
    """
    Input  : (C+1, S+1, D)   from STM
    Output : (M+1, C+1, S+1)

    FIX: self.M = num_submatrices  (was wrongly set to Cp1*Sp1, causing
         billions of parameters and shape mismatches in TemporalTFB).
    """
    def __init__(self, stm_out_shape, num_submatrices, num_blocks, num_heads):
        super().__init__()
        Cp1, Sp1, D = stm_out_shape
        self.avgf = num_submatrices
        self.M    = num_submatrices          # ← FIXED: small M, not Cp1*Sp1
        self.tK   = num_blocks
        assert D % num_submatrices == 0, f"D={D} must be divisible by num_submatrices={num_submatrices}"
        assert self.M % num_heads == 0,  f"M={self.M} must be divisible by num_heads={num_heads}"

        flat = Cp1 * Sp1
        self.weight = nn.Parameter(torch.randn(self.M, flat))        # (M, flat)
        self.bias   = nn.Parameter(torch.zeros(num_submatrices + 1, self.M))
        self.cls    = nn.Parameter(torch.zeros(1, self.M))
        trunc_normal(self.bias, std=.02)
        trunc_normal(self.cls,  std=.02)
        self.tfbs = nn.ModuleList([
            TemporalTFB(self.M, num_heads, num_submatrices)
            for _ in range(num_blocks)
        ])
        self.ln = nn.LayerNorm(self.M)
        self._Cp1, self._Sp1, self._D = Cp1, Sp1, D

    def forward(self, x):           # x : (C+1, S+1, D)
        Cp1, Sp1, D = x.shape
        seg  = D // self.avgf
        flat = Cp1 * Sp1

        xc  = x.permute(2, 0, 1)                                # (D, C+1, S+1)
        xc  = xc.reshape(self.avgf, seg, Cp1, Sp1).mean(dim=1)  # (M, C+1, S+1)
        alt = xc.reshape(self.avgf, flat)                        # (M, flat)
    
        z = torch.einsum('lm,im->il', self.weight, alt)          # (M, M)  -- wait, shape issue here too
        z = torch.cat([self.cls, z], dim=0)                      # (M+1, M)
        z = z + self.bias
        for tfb in self.tfbs:
            z = tfb(x, z)
        z = self.ln(z)                                           # (M+1, M)

        # Project back to spatial dims for the decoder
        out = torch.einsum('tm,mf->tf', z, self.weight)          # (M+1, flat)
        return out.reshape(self.avgf + 1, Cp1, Sp1)              # (M+1, C+1, S+1)


# ─── CNN Decoder ─────────────────────────────────────────────────────────────
class CNNDecoder(nn.Module):
    """
    Input  : (M+1, C+1, S+1)
    Output : (1, num_classes)
    """
    def __init__(self, ttm_out_shape, num_classes, CF_second=2):
        super().__init__()
        Mp1, Cp1, Sp1 = ttm_out_shape
        self.cvd1 = nn.Conv1d(Cp1,  1,        kernel_size=1)
        self.cvd2 = nn.Conv1d(Sp1,  CF_second, kernel_size=1)
        self.cvd3 = nn.Conv1d(Mp1,  Mp1 // 2, kernel_size=1)
        self.fc   = nn.Linear((Mp1 // 2) * CF_second, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):           # x : (M+1, C+1, S+1)
        x = x.permute(2, 1, 0)                          # (S+1, C+1, M+1)
        x = self.relu(self.cvd1(x))                     # (S+1, 1,   M+1)
        x = x.squeeze(1)                                 # (S+1, M+1)
        x = self.relu(self.cvd2(x)).transpose(0, 1)     # (M+1, CF_second)
        x = self.relu(self.cvd3(x))                     # (M+1//2, CF_second)
        x = self.fc(x.reshape(1, -1))                   # (1, num_classes)
        return x


# ─── L_SCLNet with EEGformer backbone ────────────────────────────────────────
class L_SCLNet_EEGformer(nn.Module):
    """
    L_SCLNet redesigned with an EEGformer backbone.

    Pipeline:  ODCM → RTM → STM → TTM → CNNDecoder

    Args:
        input_channels  : number of EEG channels (default 21 for TUH)
        time_steps      : number of time samples per window (e.g. 512)
        num_classes     : 2 for binary seizure / non-seizure
        kernel_size     : ODCM depthwise-conv kernel size (default 10)
        num_blocks      : transformer depth K (default 3)
        num_heads_rtm   : multi-head count for RTM  (must divide D=120)
        num_heads_stm   : multi-head count for STM  (must divide D=120)
        num_heads_ttm   : multi-head count for TTM  (must divide M=num_submatrices)
        num_submatrices : temporal sub-matrix count M (must divide D=120)
        CF_second       : second conv filter count in decoder

    Shape notes (C=21, kernel_size=10, time_steps=512):
        S   = 512 - 3*(10-1) = 485
        D   = 120  (ncf, fixed inside ODCM)
        M   = num_submatrices = 10
        TTM output : (M+1, C+1, S+1) = (11, 22, 486)
    """
    def __init__(
        self,
        input_channels  = 21,
        time_steps      = 512,
        num_classes     = 2,
        kernel_size     = 10,
        num_blocks      = 3,
        num_heads_rtm   = 6,
        num_heads_stm   = 6,
        num_heads_ttm   = 5,   # must divide num_submatrices=10  →  5 or 10
        num_submatrices = 10,
        CF_second       = 2,
    ):
        super().__init__()
        ncf = 120                                    # ODCM output depth (D)
        C   = input_channels
        D   = ncf
        S   = time_steps - 3 * (kernel_size - 1)    # temporal dim after ODCM

        assert D % num_heads_rtm == 0, \
            f"D={D} must be divisible by num_heads_rtm={num_heads_rtm}"
        assert D % num_heads_stm == 0, \
            f"D={D} must be divisible by num_heads_stm={num_heads_stm}"
        assert num_submatrices % num_heads_ttm == 0, \
            f"num_submatrices={num_submatrices} must be divisible by num_heads_ttm={num_heads_ttm}"
        assert D % num_submatrices == 0, \
            f"D={D} must be divisible by num_submatrices={num_submatrices}"

        self.odcm    = ODCM(C, kernel_size)
        self.rtm     = RTM((C, D, S),       num_blocks, num_heads_rtm)
        self.stm     = STM((S, C+1, D),     num_blocks, num_heads_stm)
        self.ttm     = TTM((C+1, S+1, D),   num_submatrices, num_blocks, num_heads_ttm)
        self.decoder = CNNDecoder((num_submatrices+1, C+1, S+1), num_classes, CF_second)

    def forward(self, x):
        # x : (B, C, T)
        outs = []
        for i in range(x.shape[0]):
            xi = self.odcm(x[i])     # (C, D, S)
            xi = self.rtm(xi)        # (S, C+1, D)
            xi = self.stm(xi)        # (C+1, S+1, D)
            xi = self.ttm(xi)        # (M+1, C+1, S+1)
            xi = self.decoder(xi)    # (1, num_classes)
            outs.append(xi)
        return torch.cat(outs, dim=0)   # (B, num_classes)


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = L_SCLNet_EEGformer(
    input_channels  = 21,
    time_steps      = WIN_SAMPLES,   # 512
    num_classes     = 2,
    kernel_size     = 10,
    num_blocks      = 3,
    num_heads_rtm   = 6,
    num_heads_stm   = 6,
    num_heads_ttm   = 6,
    num_submatrices = 6,
    CF_second       = 2,
).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

counts     = np.bincount(y_train)
class_wts  = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float32).to(device)
criterion  = nn.CrossEntropyLoss(weight=class_wts)
optimizer  = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

NUM_EPOCHS = 5
scheduler  = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-4,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1,
)

best_val_acc    = 0.0
best_model_path = "best_lsclnet_eegformer.pth"
history         = {"train_loss": [], "train_acc": [], "val_acc": []}
LOG_EPOCHS      = {1, 5, 10}

for epoch in range(1, NUM_EPOCHS + 1):
    # ── train ──────────────────────────────────────────────────────────────
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    for xb, yb in tqdm(train_loader, desc=f"Epoch {epoch:02d}", leave=False):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss   = criterion(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

    train_acc = accuracy_score(all_labels, all_preds)
    avg_loss  = total_loss / len(train_loader)
    history["train_loss"].append(avg_loss)
    history["train_acc"].append(train_acc)

    # ── validate every epoch ───────────────────────────────────────────────
    model.eval()
    val_preds, val_labels_list = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            val_preds.extend(torch.argmax(model(xb), 1).cpu().numpy())
            val_labels_list.extend(yb.numpy())

    val_acc = accuracy_score(val_labels_list, val_preds)
    val_f1  = f1_score(val_labels_list, val_preds, zero_division=0)
    history["val_acc"].append(val_acc)

    # ── log only at epochs 1, 5, 10 ───────────────────────────────────────
    if epoch in LOG_EPOCHS:
        print(f"[Epoch {epoch:02d}] loss={avg_loss:.4f} | "
              f"train_acc={train_acc*100:.2f}% | "
              f"val_acc={val_acc*100:.2f}% | "
              f"val_f1={val_f1:.4f} | "
              f"lr={scheduler.get_last_lr()[0]:.2e}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        if epoch in LOG_EPOCHS:
            print(f"  ✅ Best model saved (val_acc={val_acc*100:.2f}%)")

print(f"\n🏆  Best val acc: {best_val_acc*100:.2f}%")

Using device: cuda
Parameters: 3,791,247


[Epoch 01] loss=0.6891 | train_acc=61.42% | val_acc=58.10% | val_f1=0.6124 | lr=3.00e-04
  ✅ Best model saved (val_f1=0.6124)


[Epoch 05] loss=0.6052 | train_acc=81.08% | val_acc=77.62% | val_f1=0.8243 | lr=1.00e-04
  ✅ Best model saved (val_f1=0.8243)

🏆  Best val acc: 82.4%


In [10]:
# ─── load best model and evaluate on test set ─────────────────────────────────
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

test_preds, test_labels_list = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        test_preds.extend(torch.argmax(model(xb), 1).cpu().numpy())
        test_labels_list.extend(yb.numpy())

print("📊  Test Set Results")
print(f"   Accuracy  : {accuracy_score(test_labels_list, test_preds)*100:.2f}%")
print(f"   F1 Score  : {f1_score(test_labels_list, test_preds):.4f}")
print(f"   Precision : {precision_score(test_labels_list, test_preds):.4f}")
print(f"   Recall    : {recall_score(test_labels_list, test_preds):.4f}")
print()
print("Confusion Matrix:")
print(confusion_matrix(test_labels_list, test_preds))

📊  Test Set Results
   Accuracy  : 84.89%
   F1 Score  : 0.8125
   Precision : 0.7879
   Recall    : 0.8387

Confusion Matrix:
[[ 8013  2104]
 [ 2015 10400]]
